# Pipeline 8 : Molecules analogues, squelettes recurrents et activity cliffs

La question du chercheur : j'ai une molecule interessante, laquelle lui ressemble et que sait-on d'elle ? Et quels sont les squelettes chimiques qui reviennent chez les molecules actives ?

Ce dernier notebook rassemble trois analyses qui sortent du cadre classique et qui montrent une vraie maturite cheminformatique. D'abord un moteur de recherche de molecules analogues, l'equivalent moleculaire du systeme de recommandation. Ensuite une analyse des scaffolds de Murcko, les squelettes qui structurent les molecules. Enfin la detection des activity cliffs, ces paires de molecules quasi identiques mais aux activites opposees, qui sont a la fois le cauchemar des modeles et une mine d'or pour comprendre ce qui fait vraiment l'activite.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold

C_BLEU = "#1f6f8b"
C_ORANGE = "#e0771a"
C_VERT = "#2e8b57"
C_ROUGE = "#9b2226"
C_GRIS = "#8d99ae"

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13
})

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('egfr_descripteurs.csv')
df['activite'] = np.where(df['pIC50'] >= 6, 'actif',
                          np.where(df['pIC50'] < 5, 'inactif', 'intermediaire'))
df['mol'] = df['canonical_smiles'].apply(Chem.MolFromSmiles)
df = df[df['mol'].notnull()].reset_index(drop=True)

# Empreintes pour la similarite de Tanimoto (objets RDKit natifs)
df['fp'] = df['mol'].apply(lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048))
print(f"Molecules pretes : {len(df)}")

# 1. Moteur de recherche de molecules analogues

La similarite de Tanimoto compte la proportion de motifs structurels partages entre deux molecules. Elle vaut 1 pour deux molecules identiques et 0 pour deux molecules sans aucun motif commun. C'est la mesure de reference en cheminformatique. On construit une fonction qui, pour une molecule donnee, retrouve ses plus proches analogues et affiche leur activite connue.

In [ ]:
def trouver_analogues(smiles, n=5):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print("SMILES invalide.")
        return None
    fp_cible = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    similarites = df['fp'].apply(lambda fp: DataStructs.TanimotoSimilarity(fp_cible, fp))
    resultat = df.copy()
    resultat['similarite'] = similarites
    # On exclut la molecule elle-meme si elle est dans la base
    proches = resultat[resultat['similarite'] < 0.999].nlargest(n, 'similarite')
    return proches[['canonical_smiles', 'pIC50', 'activite', 'similarite']]

# Demo : on prend une molecule active connue et on cherche ses analogues
molecule_test = df.nlargest(1, 'pIC50')['canonical_smiles'].iloc[0]
print(f"Molecule de reference (une des plus actives) :")
print(f"  {molecule_test}")
print()
analogues = trouver_analogues(molecule_test, n=5)
print("Ses 5 analogues les plus proches :")
print(analogues.round(3).to_string(index=False))

In [ ]:
# Visualisation : la molecule cible et ses analogues cote a cote
mols_a_dessiner = [Chem.MolFromSmiles(molecule_test)] + \
                  [Chem.MolFromSmiles(s) for s in analogues['canonical_smiles']]
legendes = ['Cible'] + [f"sim {s:.2f}, pIC50 {p:.1f}"
                        for s, p in zip(analogues['similarite'], analogues['pIC50'])]

img = Draw.MolsToGridImage(mols_a_dessiner, molsPerRow=3, subImgSize=(260, 200), legends=legendes)
img

In [ ]:
print("C'est l'equivalent moleculaire d'un moteur de recommandation. On donne une molecule, le systeme retrouve celles qui lui ressemblent le plus et surtout ce qu'on sait deja de leur activite. L'usage metier est puissant : si un chercheur imagine une nouvelle molecule, il peut instantanement voir si des cousines proches ont deja ete testees et si elles etaient actives. Cela evite de retester ce qui est deja connu et oriente la synthese vers les zones prometteuses.")

# 2. Les scaffolds de Murcko : les squelettes recurrents

Le scaffold de Murcko d'une molecule est son squelette, obtenu en retirant les chaines laterales pour ne garder que les cycles et ce qui les relie. Deux molecules qui partagent le meme scaffold appartiennent a la meme serie chimique. En comptant les scaffolds les plus frequents chez les molecules actives, on identifie les squelettes gagnants a exploiter.

In [ ]:
def scaffold_de(mol):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None

df['scaffold'] = df['mol'].apply(scaffold_de)
actifs = df[df['activite'] == 'actif']

top_scaffolds = actifs['scaffold'].value_counts().head(6)
print("Les 6 squelettes les plus frequents chez les molecules actives :")
for i, (scaf, n) in enumerate(top_scaffolds.items(), 1):
    print(f"  {i}. present dans {n} molecules actives")

mols_scaf = [Chem.MolFromSmiles(s) for s in top_scaffolds.index if s and Chem.MolFromSmiles(s)]
legendes_scaf = [f"{n} molecules actives" for n in top_scaffolds.values[:len(mols_scaf)]]
img = Draw.MolsToGridImage(mols_scaf, molsPerRow=3, subImgSize=(260, 200), legends=legendes_scaf)
img

In [ ]:
print("Ces squelettes sont les fondations chimiques des inhibiteurs de l'EGFR dans notre base. Quand un squelette revient chez de nombreuses molecules actives, c'est un signal fort : ce motif structurel est compatible avec l'inhibition de la proteine. Pour l'equipe de chimie, ces scaffolds sont des points de depart concrets. On synthetise de nouvelles molecules en gardant le squelette gagnant et en variant les chaines laterales, une strategie classique et efficace appelee optimisation de tete de serie.")

# 3. Detection des activity cliffs

Un activity cliff est une paire de molecules structurellement tres similaires mais dont les activites sont tres differentes. Ce sont des cas fascinants et redoutables : ils prouvent qu'un petit changement structurel peut tout changer, et ils mettent en echec les modeles qui supposent que des molecules proches ont des activites proches. Les reperer, c'est identifier les endroits ou la relation structure activite est la plus sensible.

In [ ]:
# On cherche des paires tres similaires (Tanimoto eleve) mais aux pIC50 tres differents
import itertools

# Pour rester leger, on echantillonne un sous-ensemble de molecules bien mesurees
sous = df.sample(n=min(400, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
cliffs = []
seuil_sim = 0.7
seuil_diff = 2.0

for i, j in itertools.combinations(range(len(sous)), 2):
    sim = DataStructs.TanimotoSimilarity(sous.loc[i, 'fp'], sous.loc[j, 'fp'])
    if sim >= seuil_sim:
        diff = abs(sous.loc[i, 'pIC50'] - sous.loc[j, 'pIC50'])
        if diff >= seuil_diff:
            cliffs.append({'i': i, 'j': j, 'similarite': sim, 'diff_pIC50': diff})

df_cliffs = pd.DataFrame(cliffs).sort_values('diff_pIC50', ascending=False)
gc.collect()
print(f"Activity cliffs detectes (similarite >= {seuil_sim} et difference de pIC50 >= {seuil_diff}) : {len(df_cliffs)}")
if len(df_cliffs) > 0:
    print(df_cliffs.head(8).round(2).to_string(index=False))

In [ ]:
if len(df_cliffs) > 0:
    paire = df_cliffs.iloc[0]
    i, j = int(paire['i']), int(paire['j'])
    mols = [sous.loc[i, 'mol'], sous.loc[j, 'mol']]
    legendes = [f"pIC50 = {sous.loc[i, 'pIC50']:.1f}", f"pIC50 = {sous.loc[j, 'pIC50']:.1f}"]
    img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(320, 250), legends=legendes)
    display(img)
    print(f"Ces deux molecules se ressemblent a {paire['similarite']*100:.0f}% mais leur puissance differe de {paire['diff_pIC50']:.1f} unites de pIC50, soit un facteur {10**paire['diff_pIC50']:.0f} sur l'IC50.")
    print("C'est un activity cliff spectaculaire. Un detail structurel, souvent un seul atome ou un seul groupe, fait basculer la molecule d'active a inactive. Pour un chimiste, ces paires sont de l'or pur : elles pointent exactement l'endroit de la molecule qui gouverne l'interaction avec l'EGFR. Comprendre pourquoi ce petit changement a un si grand effet, c'est comprendre le mecanisme d'action, et c'est aussi la raison pour laquelle nos modeles de regression plafonnent, car ils ne peuvent pas deviner ces bascules brutales a partir d'empreintes presque identiques.")
else:
    print("Aucun activity cliff marque dans cet echantillon, essayer d'augmenter la taille de l'echantillon ou d'assouplir les seuils.")

# Conclusion

Ces trois analyses ajoutent une couche de finesse chimique au projet. Le moteur d'analogues transforme la base de molecules en un outil de recherche interactif, l'analyse des scaffolds identifie les squelettes porteurs sur lesquels construire, et la detection des activity cliffs eclaire a la fois les zones les plus informatives de la relation structure activite et les limites intrinseques de nos modeles.

La limite generale de ce notebook est le cout de calcul de la similarite par paires, qui grandit avec le carre du nombre de molecules, raison pour laquelle on a echantillonne pour les activity cliffs. Sur une vraie base industrielle de millions de molecules, il faudrait des structures de donnees specialisees. Mais pour un POC, ces analyses demontrent parfaitement la valeur qu'une equipe data apporte a un projet de decouverte de medicaments.